In [1]:
# No-RAG Baseline
# Converted from notebooks/4pt_noRAG.ipynb

# === Imports & Config ===
import os
import json
import re
import math
import pathlib
import itertools
import collections
import getpass
import pandas as pd
import numpy as np
from openai import OpenAI
import matplotlib.pyplot as plt
import fitz

from dotenv import load_dotenv
import os
load_dotenv()  # 自动读取当前工作目录下的 .env
api_key = os.getenv("OPENAI_API_KEY")

CODEBOOK_MD = "/Users/xinby/Desktop/AI44PT_Desktop/data/processed/TheCodingTask.md"
PAPER_PDF = "/Users/xinby/Desktop/AI44PT_Desktop/data/processed/sample_paper.pdf"
CLS_MODEL = "gpt-5-2025-08-07"

# Batch assessment settings
MAX_ITEMS = 1  # Number of items to process at a time (OpenAI API limit), None for all at once
SEED = 7          # Random seed
if not os.environ.get('OPENAI_API_KEY'):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter OPENAI_API_KEY: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client ok. Using model:", CLS_MODEL)

CODEBOOK_TEXT = pathlib.Path(CODEBOOK_MD).read_text(encoding="utf-8")
print("Loaded Codebook.md chars:", len(CODEBOOK_TEXT))


def read_pdf(path: str):
    """
    Read a PDF file and extract text from each page.
    Returns a list of dictionaries with 'page' and 'text' keys.
    """
    doc = fitz.open(path)
    pages = []
    for i, pg in enumerate(doc):
        pages.append({'page': i+1, 'text': pg.get_text('text') or ''})
    return pages


def read_markdown(path: str, heading_pattern: str = r'^#{1,6}\s'):
    """
    Read a markdown file and split it into sections based on headings.
    Each section starts with a heading matching the given pattern.
    Returns a list of dictionaries with 'page' (section index) and 'text'.
    """
    heading_re = re.compile(heading_pattern)
    sections = []
    current_lines = []
    section_idx = 1
    with open(path, 'r', encoding='utf-8') as f:
        for raw_line in f:
            line = raw_line.rstrip('\n')
            if heading_re.match(line.strip()):
                if current_lines:
                    sections.append({'page': section_idx, 'text': '\n'.join(current_lines).strip()})
                    section_idx += 1
                current_lines = [line]
            else:
                current_lines.append(line)
        if current_lines:
            sections.append({'page': section_idx, 'text': '\n'.join(current_lines).strip()})
    if not sections:
        text = pathlib.Path(path).read_text(encoding='utf-8')
        sections.append({'page': 1, 'text': text})
    return sections


# Load codebook and paper sections/pages
cb_pages = read_markdown(CODEBOOK_MD)
paper_pages = read_pdf(PAPER_PDF)
print("Codebook sections:", len(cb_pages), "Sample:")
print(cb_pages[0] if cb_pages else None)
print("Paper pages:", len(paper_pages), "Sample:")
print(paper_pages[0] if paper_pages else None)


# === 4PT Analysis Questions ===
FOURPT_QUESTIONS = """
Please analyze this article using the 4PT framework by answering the following questions:

1. Does the article fit in the universe of sustainability analyses we seek to assess? (Yes/No)

2. What problems or set of problems is the article trying to address?

3. Do the analysis, conclusions, and theories derived from, and directed to, understanding and/or managing a clearly specified 'on the ground' problem or class of problems? (Yes/No)

4. Provide arguments that support your response to Q3 (Does the article address a clearly specified on-ground problem?)

5. Provide some key text passages from the article that support your Q3 response

6. Are the analysis, conclusions, and theories generated to apply beyond understanding, and/or managing a clearly specified 'on the ground' problem or class of problems? (Yes/No)

7. Provide arguments that support your response to Q6 (Does the article generate analysis, conclusions, and theories to apply beyond understanding/managing a clearly specified on-ground problem?)

8. Provide some key text passages from the article that support your Q6 response

9. Do the analysis, conclusions, and theories treat individuals, organizations and states as largely self-interested, satisfaction driven entities that seek to maximize some kind of 'utility' outcome? (Yes/No)

10. Provide arguments that support your response to Q9 (Does the article treat entities as self-interested, utility-maximizing agents?)

11. Provide some key text passages from the article that support your Q9 response

12. Do the analysis incorporate theories and conclusions incorporate an assessment of individuals, organizations and/or states that extends beyond self-interested satisfaction seeking motivations? (Yes/No)

13. Provide arguments that support your response to Q12 (Does the article extend beyond self-interested satisfaction seeking motivations?)

14. Provide some key text passages from the article that support your Q12 response

15. Based on your analysis above, what is your final 4PT Type classification? (T1, T2, T3, or T4)

16. What is the difficulty level of this classification? (easy, medium, or hard)

Please answer each question clearly and provide specific evidence from the text when requested.
"""


def analyze_article_fourpt(article_pages: list, codebook_pages: list, model: str = CLS_MODEL):
    """
    使用4PT框架分析文章，采用原始的问答模式
    """
    # 将页面内容合并为文本
    article_text = "\n\n".join([f"Page {p['page']}:\n{p['text']}" for p in article_pages])
    codebook_text = "\n\n".join([f"Section {p['page']}:\n{p['text']}" for p in codebook_pages])
    
    prompt = f"""
You are an expert public policy analyst reviewing sustainability research articles. 

**Instructions:**
- Answer ALL questions only based on the provided Codebook and Article
- Provide specific citations when requested
- Keep justifications concise and evidence-based
- For Yes/No questions, choose definitively based on evidence

**4PT Codebook:**
{codebook_text}

**Article to Analyze:**
{article_text}

{FOURPT_QUESTIONS}
    """.strip()

    try:
        # 尝试使用新的 responses API 风格，如果不可用则回退到标准API
        try:
            # 按照用户提供的范式尝试调用
            response = client.responses.create(
                model=model,
                input=prompt,
                reasoning={"effort": "high"},
                text={"verbosity": "medium"},
            )
            analysis_text = response.output_text
        except AttributeError:
            # 如果新API不存在，使用标准的chat completions API
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                temperature=0.1
            )
            analysis_text = response.choices[0].message.content

        
        # 创建结果字典，包含原始文本和一些元数据
        result = {
            "analysis_text": analysis_text,
            "analysis_metadata": {
                "model": model,
                "timestamp": pd.Timestamp.now().isoformat(),
                "prompt_length": len(prompt),
                "response_length": len(analysis_text)
            }
        }
        
        return result
        
    except Exception as e:
        print(f"Error in analysis: {e}")
        return None


def save_analysis_result(result, output_dir="results", filename_prefix="4pt_analysis", article_name="sample_paper"):
    """
    保存分析结果为易读的txt和md格式
    """
    if not result:
        print("No result to save!")
        return
    
    # 确保输出目录存在
    os.makedirs(output_dir, exist_ok=True)
    
    # 生成文件名（包含文章名和时间戳）
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    txt_filename = f"{filename_prefix}_{article_name}_{timestamp}.txt"
    md_filename = f"{filename_prefix}_{article_name}_{timestamp}.md"
    
    txt_path = os.path.join(output_dir, txt_filename)
    md_path = os.path.join(output_dir, md_filename)
    
    # 格式化内容
    content_header = f"""4PT Framework Analysis Result
{"=" * 50}
Generated: {result['analysis_metadata']['timestamp']}
Model: {result['analysis_metadata']['model']}
Prompt Length: {result['analysis_metadata']['prompt_length']} characters
Response Length: {result['analysis_metadata']['response_length']} characters

"""
    
    # TXT格式（纯文本）
    txt_content = content_header + "ANALYSIS:\n" + "-" * 30 + "\n" + result['analysis_text']
    
    # MD格式（Markdown）- 更好的格式化
    analysis_formatted = result['analysis_text'].replace('\n\n', '\n\n')  # 保持段落间距
    
    md_content = f"""# 4PT Framework Analysis Result

## 📊 Metadata
- **Generated:** {result['analysis_metadata']['timestamp']}
- **Model:** {result['analysis_metadata']['model']}
- **Prompt Length:** {result['analysis_metadata']['prompt_length']:,} characters
- **Response Length:** {result['analysis_metadata']['response_length']:,} characters

---

## 📝 Analysis Result

{analysis_formatted}

---

## 📋 Analysis Structure
The analysis follows the 4PT framework which evaluates articles based on:
1. **Sustainability fit** - Does the article fit within sustainability analyses?
2. **Problem specification** - What problems does the article address?
3. **On-ground focus** - Does it address clearly specified real-world problems?
4. **Beyond application** - Does it generate theories applicable beyond specific problems?
5. **Utility maximization** - Does it treat entities as self-interested utility maximizers?
6. **Beyond self-interest** - Does it extend beyond self-interested motivations?
7. **4PT Classification** - Final type classification (T1, T2, T3, or T4)

---
*Generated by 4PT Analysis System v2.0 - No-RAG Baseline*
"""
    
    # 保存文件
    try:
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(txt_content)
        
        with open(md_path, 'w', encoding='utf-8') as f:
            f.write(md_content)
        
        print(f"\n✅ Results saved successfully:")
        print(f"   📄 TXT: {txt_path}")
        print(f"   📝 MD:  {md_path}")
        
        return txt_path, md_path
        
    except Exception as e:
        print(f"❌ Error saving files: {e}")
        return None, None


if __name__ == '__main__':
    result = analyze_article_fourpt(paper_pages, cb_pages)

    # 输出并保存分析结果
    if result:
        print("=" * 60)
        print("4PT ANALYSIS RESULT")
        print("=" * 60)
        print(result['analysis_text'])
        print("\n" + "=" * 60)
        print("METADATA:")
        print(f"Model: {result['analysis_metadata']['model']}")
        print(f"Timestamp: {result['analysis_metadata']['timestamp']}")
        print(f"Prompt Length: {result['analysis_metadata']['prompt_length']} chars")
        print(f"Response Length: {result['analysis_metadata']['response_length']} chars")
        
        # 保存结果到文件
        save_analysis_result(result, article_name="sample_paper")
        
    else:
        print("Analysis failed!")

        

# For reference, the human coder's answer for this paper is provided below as comments:
# 1. Yes
# 2. Interaction of program design conditions in affecting performance of voluntary programs
# 3. No
# 4. Geographical focus in several countries with no clearly specified on the ground problem
# 5. Looks at the topic of low carbon building and city development across cases in Australia, the Netherlands and the US
# 6. Yes
# 7. Explores implications on voluntary program performance of the diffusion network
# 8. This article has introduced the diffusion network perspective as a critical condition to understand voluntary program performance
# 9. Analysis on generalizability is valid. Proceed to the next questions on utility.
# 10. Yes
# 11. Key objective: determinants of voluntary program performance, which originates from high cost of direct regulatory intervention
# 12. Voluntary programs aim to change the behavior of individuals and organizations, but without the force of law. They have rapidly become popular governance instruments in situations in which it is too costly or difficult to implement direct regulatory interventions, for instance because of political unwillingness (Darnall & Carmin 2005). They also provide an opportunity to showcase and market desired “beyond compliance” behavior, or to reward leading firms (Saurwein 2011).
# 13. No
# 14. Looks at how diffusion network perspective can relate to better voluntary program results
# 15. The diffusion network perspective, as conceptualized in this article, brings together these insights and argues that the stronger the diffusion network, the more likely it is that a voluntary program will achieve the desired results.
# 16. Analysis on generalizability is valid. Proceed to the next questions on utility.
# 17. Type 2
# 18. 4 - Hard


OpenAI client ok. Using model: gpt-5-2025-08-07
Loaded Codebook.md chars: 43556
Codebook sections: 13 Sample:
{'page': 1, 'text': '# The Coding Task\n\n***This codebook develops a systematic and standardized way to classify qualitative documents into one of Cashore’s four problem types(4PT) framework*** . It will do so by providing guidance to coders on how to assign Type 1, 2, 3 or 4 status to qualitative text including, but not limited to, peer reviewed journal articles, course syllabi, policy statements, policy analyses, media communications and policy recommendations. This classification effort complements, rather than replaces, careful qualitative application\nof the framework required for uncovering important nuances through deliberative analysis and reflection of highly complex text. ***What the quantitative coding effort does well is to generate useful knowledge about whether the overall patterns identified by an one study or set of studies compare to other cases outside of the